# 01 - Análisis Exploratorio de Datos (EDA)
## Proyecto Final TI24 — Bagging: Random Forests
**Estudiante:** Fabio Mavric  
**Dataset:** Cardiovascular Disease Dataset (Kaggle)  
**Fecha:** Junio 2025

In [ ]:
# Instalación de librerías (si es necesario)
# !pip install pandas numpy matplotlib seaborn scikit-learn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Configuración visual
plt.rcParams['figure.figsize'] = (10, 6)
sns.set_style('whitegrid')
sns.set_palette('Set2')

print('✅ Librerías cargadas correctamente')

## 1. Carga del Dataset

In [ ]:
# INSTRUCCIÓN: Coloca el archivo cardio_train.csv en la carpeta data/raw/
df = pd.read_csv('../data/raw/cardio_train.csv', sep=';')

print(f'✅ Dataset cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas')
df.head()

## 2. Descripción del Dataset

| Variable | Tipo | Descripción |
|----------|------|-------------|
| age | Numérica | Edad en días |
| gender | Categórica | 1=Mujer, 2=Hombre |
| height | Numérica | Altura en cm |
| weight | Numérica | Peso en kg |
| ap_hi | Numérica | Presión sistólica |
| ap_lo | Numérica | Presión diastólica |
| cholesterol | Categórica | 1=Normal, 2=Alto, 3=Muy alto |
| gluc | Categórica | 1=Normal, 2=Alto, 3=Muy alto |
| smoke | Categórica | 0/1 |
| alco | Categórica | 0/1 |
| active | Categórica | 0/1 |
| cardio | **Target** | 0=Sin enfermedad, 1=Con enfermedad |

In [ ]:
# Información general
print('=== INFORMACIÓN GENERAL ===')
df.info()

print('\n=== PRIMERAS 5 FILAS ===')
df.head()

In [ ]:
# Convertir edad de días a años
df['age_years'] = (df['age'] / 365).round(1)

# Estadísticas descriptivas
print('=== ESTADÍSTICAS DESCRIPTIVAS ===')
df[['age_years', 'height', 'weight', 'ap_hi', 'ap_lo']].describe().round(2)

## 3. Valores Nulos

In [ ]:
# Verificación de valores nulos
nulls = df.isnull().sum()
null_pct = (nulls / len(df) * 100).round(2)

null_df = pd.DataFrame({'Nulos': nulls, 'Porcentaje': null_pct})
print('=== VALORES NULOS POR COLUMNA ===')
print(null_df[null_df['Nulos'] > 0] if null_df['Nulos'].sum() > 0 else 'No hay valores nulos')

## 4. Distribución de la Variable Objetivo

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Conteo
target_counts = df['cardio'].value_counts()
axes[0].bar(['Sin Enfermedad (0)', 'Con Enfermedad (1)'], target_counts.values,
            color=['#2ecc71', '#e74c3c'], edgecolor='white')
axes[0].set_title('Distribución de la Variable Objetivo', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Cantidad de Pacientes')
for i, v in enumerate(target_counts.values):
    axes[0].text(i, v + 200, f'{v:,}\n({v/len(df)*100:.1f}%)', ha='center', fontweight='bold')

# Pie
axes[1].pie(target_counts.values, labels=['Sin Enf.', 'Con Enf.'],
            autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'],
            startangle=90, explode=(0.05, 0.05))
axes[1].set_title('Proporción de Clases', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../docs/eda_target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráfico guardado')

## 5. Distribución de Variables Numéricas

In [ ]:
num_vars = ['age_years', 'height', 'weight', 'ap_hi', 'ap_lo']

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

labels = ['Edad (años)', 'Altura (cm)', 'Peso (kg)', 'Presión Sistólica', 'Presión Diastólica']

for i, (var, label) in enumerate(zip(num_vars, labels)):
    axes[i].hist(df[var], bins=40, color='#3498db', edgecolor='white', alpha=0.8)
    axes[i].set_title(label, fontsize=12, fontweight='bold')
    axes[i].set_xlabel(label)
    axes[i].set_ylabel('Frecuencia')
    mean_val = df[var].mean()
    axes[i].axvline(mean_val, color='red', linestyle='--', linewidth=2, label=f'Media: {mean_val:.1f}')
    axes[i].legend()

axes[-1].set_visible(False)
plt.suptitle('Distribución de Variables Numéricas', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../docs/eda_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráfico guardado')

## 6. Detección de Outliers

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

outlier_vars = [('ap_hi', 'Presión Sistólica'), ('ap_lo', 'Presión Diastólica'), ('height', 'Altura (cm)')]

for ax, (var, label) in zip(axes, outlier_vars):
    ax.boxplot(df[var], patch_artist=True,
               boxprops=dict(facecolor='#3498db', alpha=0.7),
               medianprops=dict(color='red', linewidth=2))
    ax.set_title(f'Outliers en {label}', fontsize=12, fontweight='bold')
    ax.set_ylabel(label)

plt.suptitle('Detección de Valores Atípicos (Boxplots)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../docs/eda_outliers.png', dpi=150, bbox_inches='tight')
plt.show()

# Cuantificación de outliers en presión arterial
print('=== VALORES PROBLEMÁTICOS EN PRESIÓN ===')
print(f'Presión sistólica < 0: {(df["ap_hi"] < 0).sum()}')
print(f'Presión sistólica > 250: {(df["ap_hi"] > 250).sum()}')
print(f'Presión diastólica < 0: {(df["ap_lo"] < 0).sum()}')
print(f'Presión diastólica > 200: {(df["ap_lo"] > 200).sum()}')

## 7. Variables Categóricas

In [ ]:
cat_vars = [
    ('gender', 'Género', {1: 'Mujer', 2: 'Hombre'}),
    ('cholesterol', 'Colesterol', {1: 'Normal', 2: 'Alto', 3: 'Muy Alto'}),
    ('gluc', 'Glucosa', {1: 'Normal', 2: 'Alta', 3: 'Muy Alta'}),
    ('smoke', 'Fumador', {0: 'No', 1: 'Sí'}),
    ('alco', 'Alcohol', {0: 'No', 1: 'Sí'}),
    ('active', 'Activo Físic.', {0: 'No', 1: 'Sí'})
]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()
colors = ['#e74c3c', '#2ecc71']

for i, (var, label, mapping) in enumerate(cat_vars):
    grouped = df.groupby([var, 'cardio']).size().unstack(fill_value=0)
    grouped.index = [mapping.get(k, str(k)) for k in grouped.index]
    grouped.columns = ['Sin Enf.', 'Con Enf.']
    grouped.plot(kind='bar', ax=axes[i], color=['#2ecc71', '#e74c3c'], edgecolor='white')
    axes[i].set_title(f'{label} vs Enfermedad Cardíaca', fontsize=11, fontweight='bold')
    axes[i].set_xlabel('')
    axes[i].tick_params(axis='x', rotation=0)
    axes[i].legend(fontsize=9)

plt.suptitle('Variables Categóricas vs Enfermedad Cardíaca', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../docs/eda_categorical.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráfico guardado')

## 8. Matriz de Correlación

In [ ]:
corr_matrix = df[['age_years', 'height', 'weight', 'ap_hi', 'ap_lo',
                   'cholesterol', 'gluc', 'smoke', 'alco', 'active', 'cardio']].corr()

plt.figure(figsize=(11, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdYlGn',
            mask=mask, center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Matriz de Correlación — Dataset Cardiovascular', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../docs/eda_correlation.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráfico guardado')

## 9. Resumen del EDA

In [ ]:
print('=' * 55)
print('         RESUMEN DEL ANÁLISIS EXPLORATORIO')
print('=' * 55)
print(f'📊 Total de registros: {len(df):,}')
print(f'📊 Variables: {df.shape[1]}')
print(f'🎯 Balance de clases:')
for cls, cnt in df['cardio'].value_counts().items():
    print(f'   Clase {cls}: {cnt:,} ({cnt/len(df)*100:.1f}%)')
print(f'⚠️  Requiere preprocesamiento: Sí (outliers en presión arterial)')
print(f'📝 Variables con mayor correlación con cardio: ap_hi, cholesterol, age')
print('=' * 55)
print('✅ EDA completado. Continuar con 02_preprocessing.ipynb')